In [20]:
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import json
import os
from tqdm import tqdm

In [ ]:
TEST_DIR = "PlantDoc-Dataset/test/"
CLASSMAP_PATH = "classmap.json"
BASELINE_CKPT = "runs/frontiers2023/run2/mobilenetv3small_best.pt"
FINETUNE_CKPT = "runs/finetune-pd/run1/mobilenetv3small_finetune_best.pt"
IMG_SIZE = 224
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [22]:
def load_classmap(path):
    with open(path, "r") as f:
        return json.load(f)

def build_transform(normalize=False):
    t = [transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()]
    if normalize:
        t.append(transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]))
    return transforms.Compose(t)

def load_model_safely(ckpt_path):
    """Loads model and returns (model, class_names_list)"""
    ckpt = torch.load(ckpt_path, map_location="cpu")
    
    state_dict = ckpt["model"]
    w = state_dict.get("classifier.3.weight")
    if w is None:
        w = state_dict.get("classifier.1.weight")
    
    if w is None:
        raise ValueError(f"Could not find classifier weights in {ckpt_path}")
        
    num_classes = int(w.shape[0])
    print(f"Loaded checkpoint with {num_classes} classes.")
    
    # Build Model Structure
    m = models.mobilenet_v3_small(weights=None)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    m.load_state_dict(state_dict)
    m.to(DEVICE).eval()
    
    # Return the class names stored in the checkpoint (if any)
    trained_classes = ckpt.get("classes", None)
    return m, trained_classes

In [23]:
tf = build_transform(normalize=True) 
ds_test = datasets.ImageFolder(TEST_DIR, transform=tf)
dl_test = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Ground Truth: The folder names in the Test Set
test_class_names = ds_test.classes
print(f"Test Set has {len(test_class_names)} classes.")

# Load Maps
cm = load_classmap(CLASSMAP_PATH)
pv_to_unified = cm["plantvillage_to_unified"]
pd_to_unified = cm["plantdoc_to_unified"]

Test Set has 27 classes.


In [24]:
model_base, base_classes = load_model_safely(BASELINE_CKPT)
    
if not base_classes or len(base_classes) != 38:
    base_classes = [str(i) for i in range(38)] # Dummy list if missing

correct_base = 0
total = 0

with torch.no_grad():
    for x, y in tqdm(dl_test, desc="Baseline"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        preds = model_base(x).argmax(1).cpu().numpy()
        
        for i, p_idx in enumerate(preds):
            gt_idx = y[i].item()
            
            try:
                pred_name_pv = base_classes[p_idx] # Model's label
                gt_name_pd   = test_class_names[gt_idx] # Test Set's label
                
                pred_u = pv_to_unified.get(pred_name_pv, "unknown_pred")
                gt_u   = pd_to_unified.get(gt_name_pd, "unknown_gt")
                
                if pred_u == gt_u:
                    correct_base += 1
                total += 1
            except IndexError:
                pass 

acc_base = 100.0 * correct_base / total if total > 0 else 0
print(f"Baseline Accuracy (Mapped): {acc_base:.2f}%")

Loaded checkpoint with 38 classes.


Baseline: 100%|██████████| 8/8 [00:32<00:00,  4.08s/it]

Baseline Accuracy (Mapped): 2.54%


In [25]:
model_ft, ft_train_classes = load_model_safely(FINETUNE_CKPT)
    
correct_ft = 0
total_ft = 0

with torch.no_grad():
    for x, y in tqdm(dl_test, desc="Fine-Tuned"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        preds = model_ft(x).argmax(1).cpu().numpy()
        
        for i, p_idx in enumerate(preds):
            gt_idx = y[i].item()
            
            pred_name = ft_train_classes[p_idx]
            gt_name = test_class_names[gt_idx]
            if pred_name == gt_name:
                correct_ft += 1
            total_ft += 1

acc_ft = 100.0 * correct_ft / total_ft if total_ft > 0 else 0
print(f"Fine-Tuned Accuracy: {acc_ft:.2f}%")

Loaded checkpoint with 28 classes.


Fine-Tuned: 100%|██████████| 8/8 [00:34<00:00,  4.30s/it]

Fine-Tuned Accuracy: 49.58%


In [26]:
print(f"Baseline (Zero-Shot): {acc_base:.2f}%")
print(f"Fine-Tuned (Native):  {acc_ft:.2f}%")
print(f"Improvement:          +{acc_ft - acc_base:.2f}%")

Baseline (Zero-Shot): 2.54%
Fine-Tuned (Native):  49.58%
Improvement:          +47.03%
